# Step 12 - Build the unbalanced human-validated ground_truth

> Turn the manual annotation into the first ground_truth whose labels are **exclusively human-validated**. The static analyzer was only the sampling seed - none of its labels survive here.

**Paper section:** Step 3 - Ground-truth construction / Manual annotation.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/11_manual_evaluation_with_one_occurrence_by_smell.csv` | CSV (single-occurrence variant) | Step 11 (`11_summarize_evaluation`) |
| `data/10_ground_truth_sample.json` | JSON | Step 10 (metadata + yaml) |
| `data/10b_ground_truth_resample.json` | JSON | Step 10b (metadata + yaml) |
| `data/9_ground_truth_candidates.csv` | CSV | Step 9 (metadata) |
| `data/8_dataset_enriched.csv` | CSV | Step 8 (default_branch hunt) |
| `data/downloaded_workflows/` | dir | yaml fallback corpus |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/12_ground_truth_unbalanced.json` | JSON (explicit present/absent ground_truth) | Step 12b |
| `data/step12_report.json` | JSON | audit |


In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 12 - building unbalanced human-validated ground_truth...", flush=True)
"""Step 12: Build the unbalanced human-validated ground_truth from manual annotation.

Truth = data/11_manual_evaluation_with_one_occurrence_by_smell.csv (human
consensus). The static analyzer was only the sampling seed; none of its labels
survive here. Each (file_key, smell) row must be CONCORDANT - the human decision
agrees with the sampling bucket polarity:

  positive bucket + present  -> present instance  (count from final_count)
  negative bucket + absent   -> absent  instance

Flips (positive+absent, negative+present) are removed UPSTREAM: the manual
evaluation CSV is exported concordant-only, and Step 11 emits the
single-occurrence variant this step reads. A flip must never be reassigned to
the opposite polarity (rigor is bucket-specific; see README.md /
polarity-bucket-origin), so this step ASSERTS concordance and fails fast if any
flip slips through rather than silently dropping or reassigning it.

Surviving (file, smell) pairs are grouped per file into EXPLICIT present/absent
lists (no inference from a single label list). Metadata + yaml_content are hunted
across prior steps (principle 7):
  ground_truth_sample(10) -> resample(10b) -> candidates(9) (+downloaded_workflows for yaml)
default_branch is hunted ground_truth_sample(10) -> 8_dataset_enriched.

Inputs:
  data/11_manual_evaluation_with_one_occurrence_by_smell.csv     (labels - the only source of truth)
  data/10_ground_truth_sample.json        (metadata + yaml)
  data/10b_ground_truth_resample.json     (metadata + yaml)
  data/9_ground_truth_candidates.csv      (metadata; yaml via downloaded_workflows)
  data/8_dataset_enriched.csv       (default_branch hunt)
  data/downloaded_workflows/        (yaml fallback corpus; auto-extracted)

Outputs:
  data/12_ground_truth_unbalanced.json
  data/step12_report.json
"""

import argparse
import csv
import json
import os
import sys
from pathlib import Path

# Guarded __file__ hack: make dataset_utils importable when run as a plain script
# without an installed wheel. Harmless in Jupyter (__file__ undefined). Do NOT remove.
try:
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
except NameError:
    pass

from dataset_utils.bootstrap import ensure_extracted
from dataset_utils.ground_truth_sampling import N_PER_POLARITY, SMELL_CATALOG, read_yaml
from dataset_utils.hashing import compute_file_hash
from dataset_utils.integrity import PipelineIntegrityError, assert_input_matches_report

MANUAL_EVAL_PATH = Path("data/11_manual_evaluation_with_one_occurrence_by_smell.csv")
GROUND_TRUTH_SAMPLE_PATH = Path("data/10_ground_truth_sample.json")
RESAMPLE_PATH = Path("data/10b_ground_truth_resample.json")
CANDIDATES_PATH = Path("data/9_ground_truth_candidates.csv")
ENRICHED_PATH = Path("data/8_dataset_enriched.csv")
WORKFLOWS_DIR = Path("data/downloaded_workflows")

OUTPUT_PATH = Path("data/12_ground_truth_unbalanced.json")
REPORT_PATH = Path("data/step12_report.json")

# Sampling bucket polarity -> the final_decision that agrees with it (concordant).
_EXPECTED = {"positive": "present", "negative": "absent"}
_PAPER_ID = {pascal: meta[0] for pascal, meta in SMELL_CATALOG.items()}


def _file_key(repository: str, file_path: str) -> str:
    return repository.replace("/", "@") + "@" + os.path.basename(file_path)


def _int_or(value, fallback: int = 0) -> int:
    # Parse via float first so spreadsheet/pandas float-serialized integers
    # ("2.0", "1.0") coerce correctly instead of raising -> falling back to 0.
    try:
        return int(float(str(value).strip()))
    except (TypeError, ValueError):
        return fallback


def _load_metadata_sources():
    """file_key -> metadata lookups across prior steps (best source first)."""
    with GROUND_TRUTH_SAMPLE_PATH.open(encoding="utf-8") as f:
        sample = {e["file_key"]: e for e in json.load(f)}
    with RESAMPLE_PATH.open(encoding="utf-8") as f:
        resample = {e["file_key"]: e for e in json.load(f)}
    candidates: dict[str, dict] = {}
    with CANDIDATES_PATH.open(encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            candidates.setdefault(row["file_key"], row)
    # default_branch hunt from Step 8 (BOM on first column -> utf-8-sig); dedup by file_key.
    default_branch: dict[str, str] = {}
    with ENRICHED_PATH.open(encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            repo, fp = row.get("repository", ""), row.get("file_path", "")
            if repo and fp:
                default_branch.setdefault(_file_key(repo, fp), row.get("repo_default_branch", ""))
    return sample, resample, candidates, default_branch


def _metadata_for(fk, sample, resample, candidates, default_branch, prov) -> dict:
    """Assemble entry metadata + yaml_content from the first source that has each field."""
    s = sample.get(fk)
    r = resample.get(fk)
    c = candidates.get(fk)

    def pick(field):
        for src in (s, r, c):
            if src and src.get(field) not in (None, ""):
                return src[field]
        return None

    if s:
        prov["metadata_source"]["ground_truth_sample"] += 1
    elif r:
        prov["metadata_source"]["resample"] += 1
    elif c:
        prov["metadata_source"]["candidates"] += 1
    else:
        prov["metadata_source"]["MISSING"] += 1

    # yaml_content: prefer embedded; else read from downloaded_workflows by file_key.
    if s and s.get("yaml_content"):
        yaml_content = s["yaml_content"]
        prov["yaml_source"]["ground_truth_sample"] += 1
    elif r and r.get("yaml_content"):
        yaml_content = r["yaml_content"]
        prov["yaml_source"]["resample"] += 1
    else:
        yaml_content = read_yaml(fk)
        if yaml_content is None:
            yaml_content = ""
            prov["yaml_source"]["MISSING"] += 1
        else:
            prov["yaml_source"]["downloaded_workflows"] += 1

    # default_branch: ground_truth_sample first, else Step 8 hunt.
    if s and s.get("default_branch"):
        db = s["default_branch"]
        prov["default_branch_source"]["ground_truth_sample"] += 1
    elif default_branch.get(fk):
        db = default_branch[fk]
        prov["default_branch_source"]["step8_enriched"] += 1
    else:
        db = ""
        prov["default_branch_source"]["MISSING"] += 1

    return {
        "file_key": fk,
        "repository": pick("repository") or "",
        "file_path": pick("file_path") or "",
        "yaml_content": yaml_content,
        "file_line_count": _int_or(pick("file_line_count")),
        "workflow_run_count": _int_or(pick("workflow_run_count")),
        "repo_language": pick("repo_language") or "",
        "default_branch": db,
    }


def run(
    eval_path: Path = MANUAL_EVAL_PATH,
    output_path: Path = OUTPUT_PATH,
    report_path: Path = REPORT_PATH,
) -> tuple[Path, dict]:
    ensure_extracted(WORKFLOWS_DIR)  # yaml fallback corpus (no-op if already present)
    # --- Upstream integrity: verify inputs against their producing-step reports ---
    _step11_report = Path("data/step11_report.json")
    if not _step11_report.exists():
        raise PipelineIntegrityError(
            f"step 12: upstream report missing: {_step11_report}. Run Step 11 first."
        )
    with _step11_report.open(encoding="utf-8") as _rf:
        _expected_filtered = json.load(_rf).get("filtered_sha256")
    _eval_sha = compute_file_hash(eval_path)
    if not _expected_filtered or _eval_sha != _expected_filtered:
        raise PipelineIntegrityError(
            "step 12: manual-evaluation SHA drift.\n"
            f"  input file : {eval_path}\n"
            f"  current sha: {_eval_sha}\n"
            f"  expected   : {_expected_filtered} (filtered_sha256 from {_step11_report})\n"
            "Re-run Step 11 to regenerate the filtered evaluation CSV."
        )
    assert_input_matches_report(GROUND_TRUTH_SAMPLE_PATH, Path("data/step10_report.json"), "step 12: ground_truth_sample")
    assert_input_matches_report(RESAMPLE_PATH, Path("data/step10b_report.json"), "step 12: resample")
    sample, resample, candidates, default_branch = _load_metadata_sources()

    with eval_path.open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))

    # 1. Build present/absent from concordant rows. Input must be concordant
    #    (flips removed at export + by Step 11); any flip is a contract violation
    #    and aborts the build - a flip must never be reassigned to the opposite
    #    polarity.
    per_file_present: dict[str, dict[str, int]] = {}
    per_file_absent: dict[str, set] = {}
    flips: list[dict] = []
    count_warnings: list[dict] = []
    kept_present = kept_absent = 0

    for row in rows:
        fk = row["file_key"]
        smell = row["target_smell"]
        polarity = row["tool_output"]
        label = row["final_decision"]

        if _EXPECTED.get(polarity) != label:
            flips.append(
                {"file_key": fk, "smell": smell, "tool_output": polarity, "final_decision": label}
            )
            continue

        if label == "present":
            cnt = _int_or(row.get("final_count"), fallback=0)
            if cnt <= 0:
                # Clean data should never hit this (bad rows are pruned upstream).
                count_warnings.append(
                    {"file_key": fk, "smell": smell, "raw_count": row.get("final_count", "")}
                )
                cnt = 1
            per_file_present.setdefault(fk, {})[smell] = cnt
            kept_present += 1
        else:  # absent
            per_file_absent.setdefault(fk, set()).add(smell)
            kept_absent += 1

    # Contract assertion: the input is expected concordant-only. Fail fast rather
    # than silently dropping/reassigning, so an export regression cannot corrupt
    # the ground_truth (see polarity-bucket-origin rule).
    if flips:
        raise PipelineIntegrityError(
            f"step 12: input {eval_path} expected concordant (flips are removed at "
            f"export and by Step 11), but found {len(flips)} flip row(s) where "
            f"final_decision disagrees with the tool_output bucket. A flip must not be "
            f"reassigned to the opposite polarity. First: {flips[:10]}"
        )

    # 2. One entry per file with >=1 surviving (file, smell) pair.
    prov = {
        "metadata_source": {"ground_truth_sample": 0, "resample": 0, "candidates": 0, "MISSING": 0},
        "yaml_source": {"ground_truth_sample": 0, "resample": 0, "downloaded_workflows": 0, "MISSING": 0},
        "default_branch_source": {"ground_truth_sample": 0, "step8_enriched": 0, "MISSING": 0},
    }
    file_keys = sorted(set(per_file_present) | set(per_file_absent))
    ground_truth: list[dict] = []
    for fk in file_keys:
        entry = _metadata_for(fk, sample, resample, candidates, default_branch, prov)
        present = per_file_present.get(fk, {})
        absent = per_file_absent.get(fk, set())
        entry["present_smells"] = [
            {"pascal": p, "count": present[p]}
            for p in sorted(present, key=lambda s: _PAPER_ID.get(s, 999))
        ]
        entry["absent_smells"] = [
            {"pascal": p} for p in sorted(absent, key=lambda s: _PAPER_ID.get(s, 999))
        ]
        ground_truth.append(entry)

    # 3. Per-smell, per-polarity tallies (audit deficits vs paper target N).
    per_smell: dict[str, dict] = {}
    for smells in per_file_present.values():
        for s in smells:
            per_smell.setdefault(s, {"present": 0, "absent": 0})["present"] += 1
    for smells in per_file_absent.values():
        for s in smells:
            per_smell.setdefault(s, {"present": 0, "absent": 0})["absent"] += 1
    per_smell = {s: per_smell[s] for s in sorted(per_smell, key=lambda s: _PAPER_ID.get(s, 999))}
    deficits = {
        s: {
            "present_deficit": max(0, N_PER_POLARITY - v["present"]),
            "absent_deficit": max(0, N_PER_POLARITY - v["absent"]),
        }
        for s, v in per_smell.items()
        if v["present"] < N_PER_POLARITY or v["absent"] < N_PER_POLARITY
    }

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(ground_truth, f, indent=2, ensure_ascii=False)

    report = {
        "variant": "unbalanced",
        "n_per_polarity_target": N_PER_POLARITY,
        "n_rows_processed": len(rows),
        "kept_present": kept_present,
        "kept_absent": kept_absent,
        "input_concordant": True,
        "n_entries": len(ground_truth),
        "per_smell": per_smell,
        "deficits_vs_target": deficits,
        "provenance": prov,
        "count_warnings": count_warnings,
        "input_sha256": {
            "manual_evaluation": compute_file_hash(eval_path),
            "ground_truth_sample": compute_file_hash(GROUND_TRUTH_SAMPLE_PATH),
            "resample": compute_file_hash(RESAMPLE_PATH),
            "candidates": compute_file_hash(CANDIDATES_PATH),
            "enriched": compute_file_hash(ENRICHED_PATH),
        },
        "output_file": str(output_path),
        "output_sha256": compute_file_hash(output_path),
    }
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    return output_path, report


def _parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Build the unbalanced human-validated ground_truth from manual annotation."
    )
    p.add_argument("--input", type=Path, default=MANUAL_EVAL_PATH)
    p.add_argument("--output", type=Path, default=OUTPUT_PATH)
    p.add_argument("--report", type=Path, default=REPORT_PATH)
    return p.parse_args()


def main() -> None:
    args = _parse_args()
    output_path, report = run(args.input, args.output, args.report)
    print("Unbalanced human-validated ground_truth written.")
    print(f"  Output : {output_path}")
    print(f"  Report : {args.report}")
    print("-" * 48)
    print(f"  Rows processed : {report['n_rows_processed']}")
    print(f"  Kept present   : {report['kept_present']}")
    print(f"  Kept absent    : {report['kept_absent']}")
    print(f"  Entries        : {report['n_entries']}")
    print(f"  Smells < N={report['n_per_polarity_target']} : {len(report['deficits_vs_target'])}")
    if report["count_warnings"]:
        print(f"  WARN: {len(report['count_warnings'])} present rows had no usable count")


main()

Step 12 - building unbalanced human-validated ground_truth...


[bootstrap] data/downloaded_workflows already present — skipping extract.


Unbalanced human-validated ground_truth written.
  Output : data/12_ground_truth_unbalanced.json
  Report : data/step12_report.json
------------------------------------------------
  Rows processed : 1769
  Kept present   : 825
  Kept absent    : 944
  Entries        : 875
  Smells < N=50 : 10


---

**Next step:** [`12b_balanced_ground_truth.ipynb`](./12b_balanced_ground_truth.ipynb) - cap each (smell, polarity) bucket to N and write the final balanced ground_truth.
